In [ ]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Models
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from imblearn.combine import SMOTEENN

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:
df = pd.read_csv("NSL-KDD.csv")
df.head()

In [ ]:

print("Total Columns:", df.shape)

In [ ]:
print(df.columns)

In [ ]:
print(df.dtypes)

In [ ]:
print(df["class"].unique())

In [ ]:
# Normal type
normal_types = df[df["class"] == "normal"]["class"].unique()

# Attack types
attack_types = df[df["class"] != "normal"]["class"].unique()

print(" Normal Types:")
print(normal_types)

print("\n Attack Types:")
print(attack_types)

In [ ]:
normal = (df['class'] == "normal").sum()
anomaly = (df['class'] == "anomaly").sum()

print("Total normal Data:",normal )
print("Total anomaly Data:", anomaly)

In [ ]:

labels = ['normal', 'anomaly']
values = [normal, anomaly]

plt.figure(figsize=(6,4))
plt.bar(labels, values)

# value show on bars
for i, v in enumerate(values):
    plt.text(i, v + (max(values)*0.01), str(v), ha='center')

plt.title("Before SMOTE: Normal vs Anomaly")
plt.ylabel("Count")
plt.show()

In [ ]:
anomaly_data = df[df["class"] == "anomaly"]

anomaly_data

In [ ]:

labels = ['normal', 'anomaly']
values = [normal, anomaly]

plt.figure(figsize=(6,4))
plt.bar(labels, values)

# value show on bars
for i, v in enumerate(values):
    plt.text(i, v + (max(values)*0.01), str(v), ha='center')

plt.title("Before SMOTE: Normal vs Anomaly")
plt.ylabel("Count")
plt.show()

In [ ]:
print("Missing Values in Each Column")

print(df.isnull().sum())

In [ ]:
print("Total Missing Values:", df.isnull().sum().sum())

In [ ]:
duplicates = df.duplicated().sum()

print("Total Duplicate Rows:", duplicates)

In [ ]:
print("Before removing duplicates:", df.shape)

In [ ]:
df = df.drop_duplicates()

In [ ]:


print("Duplicate rows:", df.duplicated().sum())

In [ ]:
# After removing duplicate value
print(df["class"].value_counts())

In [ ]:
import matplotlib.pyplot as plt

# Direct dataframe se values lena (best practice)
counts = df['class'].value_counts()

labels = counts.index.tolist()
values = counts.values.tolist()

plt.figure(figsize=(6,4))
plt.bar(labels, values)

# value show on bars
for i, v in enumerate(values):
    plt.text(i, v + (max(values)*0.01), str(v), ha='center')

plt.title("After Removing Duplicates (Before SMOTE)")
plt.ylabel("Count")
plt.xlabel("Class")

plt.show()

In [ ]:
#   TARGET PROCESSING (class column)
# ============================================
# Convert multi-class to binary
if df['class'].dtype == 'object':
    df['class'] = df['class'].apply(lambda x: 0 if x == 'normal' else 1)



In [ ]:
# 3. ENCODING (IMPORTANT)
# ===============================
le = LabelEncoder()

for col in df.select_dtypes(include=['object']).columns:
    df[col] = le.fit_transform(df[col])


In [ ]:
# Separate features & target
X = df.drop("class", axis=1)
y = df["class"]
print("Total Input Features:", X.shape)
feature_names = X.columns

In [ ]:
# ============================================
# TRAIN-TEST SPLIT (FIRST SPLIT)
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

print("\nTraining class distribution:")
print(y_train.value_counts())

print("\nTesting class distribution:")
print(y_test.value_counts())


In [ ]:
#  FEATURE SCALING
# ============================================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# ============================================
# MODEL DEFINITION
# ============================================

models = {
    "Decision Tree": DecisionTreeClassifier(
        random_state=RANDOM_STATE
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "KNN": KNeighborsClassifier(
        n_neighbors=5
    ),

    "SVM": LinearSVC(
        random_state=RANDOM_STATE,
        max_iter=5000
    ),

    "XGBoost": XGBClassifier(
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "LightGBM": LGBMClassifier(
        random_state=RANDOM_STATE,
        verbosity=-1,
        n_jobs=-1
    )
}


In [ ]:
# ============================================
# BASELINE MODEL EVALUATION
# ============================================

def get_model_scores(model, X_data):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X_data)[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(X_data)
    return None

before_results = {}
before_predictions = {}

for name, model in models.items():

    start = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start

    start = time.time()
    pred = model.predict(X_test)
    pred_time = time.time() - start

    acc = accuracy_score(y_test, pred)
    precision = precision_score(y_test, pred, average="weighted", zero_division=0)
    recall = recall_score(y_test, pred, average="weighted", zero_division=0)
    f1 = f1_score(y_test, pred, average="weighted", zero_division=0)

    scores = get_model_scores(model, X_test)
    roc_auc = roc_auc_score(y_test, scores) if scores is not None else np.nan

    before_results[name] = {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "train_time": train_time,
        "prediction_time": pred_time
    }

    before_predictions[name] = pred

    print(f"\n{name}")
    print("Accuracy :", round(acc, 4))
    print("Precision:", round(precision, 4))
    print("Recall   :", round(recall, 4))
    print("F1 Score :", round(f1, 4))
    print("ROC-AUC  :", round(roc_auc, 4) if not np.isnan(roc_auc) else "N/A")
    print("Train Time:", round(train_time, 4), "sec")
    print("Prediction Time:", round(pred_time, 4), "sec")


In [ ]:
# ============================================
# BASELINE EXECUTION TIME
# ============================================

models_list = list(before_results.keys())

train_time = [
    before_results[m]["train_time"]
    for m in models_list
]

pred_time = [
    before_results[m]["prediction_time"]
    for m in models_list
]

plt.figure(figsize=(8, 5))

plt.plot(
    models_list,
    train_time,
    marker="o",
    label="Training Time"
)

plt.plot(
    models_list,
    pred_time,
    marker="o",
    label="Prediction Time"
)

plt.xlabel("Models")
plt.ylabel("Time (seconds)")
plt.title("Baseline IDS Execution Time Comparison")
plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.grid(True)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


In [ ]:
#   SMOTEENN (ONLY TRAIN DATA)
# ============================================
smote_enn = SMOTEENN(random_state=42)
X_res, y_res = smote_enn.fit_resample(X_train, y_train)
print("After SMOTE-ENN:", X_res.shape)

In [ ]:
print("After SMOTE-ENN class distribution:\n")
print("Normal:", (y_res == 0).sum())
print("anomaly:", (y_res == 1).sum())

In [ ]:


# Convert to pandas Series (if needed)
y_res = pd.Series(y_res)

# Replace 0/1 with names
labels = y_res.map({0: 'Normal', 1: 'Anomaly'})

# Count
counts = labels.value_counts()

# Plot
plt.figure(figsize=(6,4))
plt.bar(counts.index, counts.values)

# Show values on bars
for i, v in enumerate(counts.values):
    plt.text(i, v + (max(counts.values)*0.01), str(v), ha='center')

plt.title("After SMOTE: Normal vs Anomaly")
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()

In [ ]:
# ============================================
# EBBA VALIDATION SPLIT
# ============================================

X_ebba_train, X_ebba_val, y_ebba_train, y_ebba_val = train_test_split(
    X_res,
    y_res,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_res
)

print("EBBA Training Data:", X_ebba_train.shape)
print("EBBA Validation Data:", X_ebba_val.shape)


In [ ]:
# ============================================
# EBBA - IMPROVED VERSION
# ============================================

n_bats = 15
n_iter = 20
n_features = X_res.shape[1]

fmin, fmax = 0, 2
A = 0.8
r = 0.5

np.random.seed(RANDOM_STATE)

bats = np.random.randint(
    2,
    size=(n_bats, n_features)
)

velocity = np.zeros(
    (n_bats, n_features)
)

best_bat = bats[0].copy()
best_score = 0.0

convergence = []

# ============================================
# VALIDATION-BASED FITNESS FUNCTION
# ============================================

def fitness_function(features):

    if features.sum() == 0:
        return 0.0

    selected = features.astype(bool)

    X_train_selected = X_ebba_train[:, selected]
    X_val_selected = X_ebba_val[:, selected]

    model = DecisionTreeClassifier(
        max_depth=5,
        random_state=RANDOM_STATE
    )

    model.fit(
        X_train_selected,
        y_ebba_train
    )

    val_accuracy = model.score(
        X_val_selected,
        y_ebba_val
    )

    feature_penalty = features.sum() / n_features

    return val_accuracy - 0.01 * feature_penalty


# ============================================
# EBBA SEARCH
# ============================================

for t in range(n_iter):

    for i in range(n_bats):

        freq = (
            fmin
            + (fmax - fmin) * np.random.rand()
        )

        velocity[i] = (
            velocity[i]
            + (bats[i] - best_bat) * freq
        )

        sigmoid = 1 / (
            1 + np.exp(-velocity[i])
        )

        new_bat = (
            np.random.rand(n_features) < sigmoid
        ).astype(int)

        # Mutation
        mutation_prob = 0.10

        for j in range(n_features):
            if np.random.rand() < mutation_prob:
                new_bat[j] = 1 - new_bat[j]

        # Local search
        if np.random.rand() > r:

            new_bat = best_bat.copy()

            flip_count = min(3, n_features)

            flip = np.random.choice(
                n_features,
                flip_count,
                replace=False
            )

            new_bat[flip] = 1 - new_bat[flip]

        score = fitness_function(new_bat)

        if score > best_score:

            bats[i] = new_bat.copy()
            best_score = score
            best_bat = new_bat.copy()

    convergence.append(best_score)

    print(
        f"Iteration {t + 1:02d}/{n_iter} | "
        f"Best Fitness: {best_score:.4f} | "
        f"Selected Features: {best_bat.sum()}"
    )

print("\nFinal EBBA Fitness:", round(best_score, 4))
print("Final Selected Features:", int(best_bat.sum()))


In [ ]:
# ============================================
# EBBA CONVERGENCE CURVE
# ============================================

plt.figure(figsize=(8, 5))

plt.plot(
    range(1, len(convergence) + 1),
    convergence,
    marker="o"
)

plt.xlabel("Iteration")
plt.ylabel("Best Fitness")
plt.title("EBBA Convergence Curve")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:


# select features
X_train_sel = X_res[:, best_bat == 1]
X_test_sel = X_test[:, best_bat == 1]

print("Selected Features:", best_bat.sum())
selected_features = X.columns[np.where(best_bat == 1)[0]]

print("\n Selected Features by EBBA:\n")

for i, feature in enumerate(selected_features, 1):
    print(f"{i}. {feature}")

print("\nTotal Selected Features:", len(selected_features))

In [ ]:
# ============================================
# EBBA SELECTED FEATURES
# ============================================

selected_mask = best_bat.astype(bool)

X_train_sel = X_res[:, selected_mask]
X_test_sel = X_test[:, selected_mask]

selected_features = X.columns[selected_mask]

print("Selected Features:", len(selected_features))

print("\nSelected Features by EBBA:\n")

for i, feature in enumerate(selected_features, 1):
    print(f"{i}. {feature}")

print("\nTotal Original Features:", X.shape[1])
print("Total Selected Features:", len(selected_features))
print(
    "Feature Reduction:",
    round((1 - len(selected_features) / X.shape[1]) * 100, 2),
    "%"
)


In [ ]:
# ============================================
# EBBA FEATURE REDUCTION
# ============================================

total_features = X.shape[1]
selected_count = len(selected_features)

plt.figure(figsize=(6, 5))

plt.bar(
    ["Before EBBA", "After EBBA"],
    [total_features, selected_count]
)

plt.ylabel("Number of Features")
plt.title("Feature Reduction Using EBBA")

for i, value in enumerate([total_features, selected_count]):
    plt.text(i, value, str(value), ha="center")

plt.tight_layout()
plt.show()


In [ ]:
# ============================================
# MODEL DEFINITION
# ============================================

models = {
    "Decision Tree": DecisionTreeClassifier(
        random_state=RANDOM_STATE
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "KNN": KNeighborsClassifier(
        n_neighbors=5
    ),

    "SVM": LinearSVC(
        random_state=RANDOM_STATE,
        max_iter=5000
    ),

    "XGBoost": XGBClassifier(
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "LightGBM": LGBMClassifier(
        random_state=RANDOM_STATE,
        verbosity=-1,
        n_jobs=-1
    )
}


In [ ]:
# ============================================
# AFTER EBBA MODEL EVALUATION
# ============================================

after_results = {}
after_predictions = {}

for name, model in models.items():

    start = time.time()
    model.fit(X_train_sel, y_res)
    train_time = time.time() - start

    start = time.time()
    pred = model.predict(X_test_sel)
    pred_time = time.time() - start

    acc = accuracy_score(y_test, pred)
    precision = precision_score(y_test, pred, average="weighted", zero_division=0)
    recall = recall_score(y_test, pred, average="weighted", zero_division=0)
    f1 = f1_score(y_test, pred, average="weighted", zero_division=0)

    scores = get_model_scores(model, X_test_sel)
    roc_auc = roc_auc_score(y_test, scores) if scores is not None else np.nan

    after_results[name] = {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "train_time": train_time,
        "prediction_time": pred_time
    }

    after_predictions[name] = pred

    print(f"\n{name}")
    print("Accuracy :", round(acc, 4))
    print("Precision:", round(precision, 4))
    print("Recall   :", round(recall, 4))
    print("F1 Score :", round(f1, 4))
    print("ROC-AUC  :", round(roc_auc, 4) if not np.isnan(roc_auc) else "N/A")
    print("Train Time:", round(train_time, 4), "sec")
    print("Prediction Time:", round(pred_time, 4), "sec")


In [ ]:
# ============================================
# IDS MODEL COMPARISON - ACCURACY
# ============================================

models_list = list(after_results.keys())

values = [
    after_results[m]["accuracy"]
    for m in models_list
]

plt.figure(figsize=(10, 5))
plt.bar(models_list, values)

for i, v in enumerate(values):
    plt.text(i, v, f"{v:.4f}", ha="center")

plt.title("IDS Model Comparison - Accuracy")
plt.xlabel("Models")
plt.ylabel("Accuracy")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================
# IDS MODEL COMPARISON - PRECISION
# ============================================

models_list = list(after_results.keys())

values = [
    after_results[m]["precision"]
    for m in models_list
]

plt.figure(figsize=(10, 5))
plt.bar(models_list, values)

for i, v in enumerate(values):
    plt.text(i, v, f"{v:.4f}", ha="center")

plt.title("IDS Model Comparison - Precision")
plt.xlabel("Models")
plt.ylabel("Precision")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================
# IDS MODEL COMPARISON - RECALL
# ============================================

models_list = list(after_results.keys())

values = [
    after_results[m]["recall"]
    for m in models_list
]

plt.figure(figsize=(10, 5))
plt.bar(models_list, values)

for i, v in enumerate(values):
    plt.text(i, v, f"{v:.4f}", ha="center")

plt.title("IDS Model Comparison - Recall")
plt.xlabel("Models")
plt.ylabel("Recall")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================
# IDS MODEL COMPARISON - F1 SCORE
# ============================================

models_list = list(after_results.keys())

values = [
    after_results[m]["f1"]
    for m in models_list
]

plt.figure(figsize=(10, 5))
plt.bar(models_list, values)

for i, v in enumerate(values):
    plt.text(i, v, f"{v:.4f}", ha="center")

plt.title("IDS Model Comparison - F1 Score")
plt.xlabel("Models")
plt.ylabel("F1 Score")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================
# AFTER EBBA EXECUTION TIME
# ============================================

models_list = list(after_results.keys())

train_time = [
    after_results[m]["train_time"]
    for m in models_list
]

pred_time = [
    after_results[m]["prediction_time"]
    for m in models_list
]

plt.figure(figsize=(8, 5))
plt.plot(models_list, train_time, marker="o", label="Training Time")
plt.plot(models_list, pred_time, marker="o", label="Prediction Time")

plt.xlabel("Models")
plt.ylabel("Time (seconds)")
plt.title("After EBBA Training and Prediction Time")
plt.xticks(rotation=30)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================
# CONFUSION MATRICES - ALL AFTER-EBBA MODELS
# ============================================

for name, pred in after_predictions.items():

    cm = confusion_matrix(y_test, pred)

    print(f"\n{name}")
    print(cm)

    plt.figure(figsize=(5, 4))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues"
    )

    plt.title(f"Confusion Matrix - {name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()


In [ ]:
# ============================================
# BEFORE vs AFTER EBBA TIME COMPARISON
# ============================================

print("\n=========== TIME COMPARISON (BEFORE vs AFTER EBBA) ===========\n")

for name in before_results:

    train_before = before_results[name]["train_time"]
    pred_before = before_results[name]["prediction_time"]

    train_after = after_results[name]["train_time"]
    pred_after = after_results[name]["prediction_time"]

    print(name)
    print(f"Train Time  (Before): {train_before:.4f} sec")
    print(f"Train Time  (After) : {train_after:.4f} sec")
    print(f"Predict Time(Before): {pred_before:.4f} sec")
    print(f"Predict Time(After) : {pred_after:.4f} sec")
    print("-" * 50)


In [ ]:
# ============================================
# BEFORE vs AFTER EBBA TIME GRAPH
# ============================================

models_list = list(before_results.keys())

train_before = [before_results[m]["train_time"] for m in models_list]
pred_before = [before_results[m]["prediction_time"] for m in models_list]

train_after = [after_results[m]["train_time"] for m in models_list]
pred_after = [after_results[m]["prediction_time"] for m in models_list]

x = np.arange(len(models_list))

plt.figure(figsize=(10, 6))

plt.plot(x, train_before, marker="o", label="Train Before EBBA")
plt.plot(x, train_after, marker="o", label="Train After EBBA")

plt.plot(x, pred_before, marker="s", linestyle="--", label="Predict Before EBBA")
plt.plot(x, pred_after, marker="s", linestyle="--", label="Predict After EBBA")

plt.xticks(x, models_list, rotation=20)
plt.xlabel("Models")
plt.ylabel("Time (seconds)")
plt.title("Training & Prediction Time: Before vs After EBBA")
plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================
# BEFORE vs AFTER EBBA - METRIC COMPARISON
# ============================================

comparison_rows = []

for name in models.keys():

    comparison_rows.append({
        "Model": name,
        "Accuracy Before": before_results[name]["accuracy"],
        "Accuracy After": after_results[name]["accuracy"],
        "Precision Before": before_results[name]["precision"],
        "Precision After": after_results[name]["precision"],
        "Recall Before": before_results[name]["recall"],
        "Recall After": after_results[name]["recall"],
        "F1 Before": before_results[name]["f1"],
        "F1 After": after_results[name]["f1"],
        "ROC-AUC Before": before_results[name]["roc_auc"],
        "ROC-AUC After": after_results[name]["roc_auc"]
    })

comparison_df = pd.DataFrame(comparison_rows)

display(comparison_df.round(4))


In [ ]:
# ============================================
# BEFORE vs AFTER EBBA METRICS
# ============================================

models_list = list(models.keys())
x = np.arange(len(models_list))
width = 0.35

before_f1 = [before_results[m]["f1"] for m in models_list]
after_f1 = [after_results[m]["f1"] for m in models_list]

plt.figure(figsize=(10, 5))

plt.bar(
    x - width / 2,
    before_f1,
    width,
    label="Before EBBA"
)

plt.bar(
    x + width / 2,
    after_f1,
    width,
    label="After EBBA"
)

plt.xticks(x, models_list, rotation=30)
plt.ylabel("Weighted F1 Score")
plt.xlabel("Models")
plt.title("Weighted F1 Score: Before vs After EBBA")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ============================================
# ROC-AUC COMPARISON
# ============================================

models_list = list(models.keys())

before_auc = [before_results[m]["roc_auc"] for m in models_list]
after_auc = [after_results[m]["roc_auc"] for m in models_list]

x = np.arange(len(models_list))
width = 0.35

plt.figure(figsize=(10, 5))

plt.bar(x - width / 2, before_auc, width, label="Before EBBA")
plt.bar(x + width / 2, after_auc, width, label="After EBBA")

plt.xticks(x, models_list, rotation=30)
plt.ylabel("ROC-AUC")
plt.xlabel("Models")
plt.title("ROC-AUC: Before vs After EBBA")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ============================================
# RANDOM FOREST FEATURE IMPORTANCE
# ============================================

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf.fit(X_train_sel, y_res)

feature_importance = pd.DataFrame({
    "Feature": selected_features,
    "Importance": rf.feature_importances_
}).sort_values(
    "Importance",
    ascending=False
)

display(feature_importance)

top_features = feature_importance.head(15)

plt.figure(figsize=(10, 6))

plt.barh(
    top_features["Feature"],
    top_features["Importance"]
)

plt.gca().invert_yaxis()

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top EBBA-Selected Feature Importance")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================
# CLASSIFICATION REPORTS
# ============================================

for name, pred in after_predictions.items():

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    print(
        classification_report(
            y_test,
            pred,
            target_names=["Normal", "Attack"],
            zero_division=0
        )
    )


In [ ]:
# ============================================
# 5-FOLD STRATIFIED CROSS-VALIDATION
# ============================================
# This uses the original training data only.
# Test data remains untouched.

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

cv_rows = []

for name, model in models.items():

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="f1_weighted",
        n_jobs=-1
    )

    cv_rows.append({
        "Model": name,
        "Mean F1": scores.mean(),
        "Std F1": scores.std()
    })

cv_df = pd.DataFrame(cv_rows)

display(cv_df.round(4))
